# House Price Prediction — Project 2

Uses the assigned Kaggle dataset. Save its CSV as `data/house_prices.csv` before running.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
DATA_PATH=Path('data/house_prices.csv')
MODEL_PATH=Path('models/house_price_best_model.joblib')
RANDOM_STATE=42


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError('Place the assigned Kaggle CSV at data/house_prices.csv first.')
df=pd.read_csv(DATA_PATH)
print('Shape:',df.shape)
display(df.head())
display(df.describe(include='all').T)
display(df.isna().sum().sort_values(ascending=False).head(20))
print('Duplicate rows:',df.duplicated().sum())


In [ ]:
common_targets=['SalePrice','saleprice','sale_price','Price','price','HousePrice','house_price','Target','target']
TARGET_COLUMN=next((c for c in common_targets if c in df.columns),None)
if TARGET_COLUMN is None:
    print(df.columns.tolist())
    TARGET_COLUMN=input('Enter target column exactly: ').strip()
print('Target:',TARGET_COLUMN)

fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.histplot(df[TARGET_COLUMN].dropna(),kde=True,ax=ax[0])
ax[0].set_title('House Price Distribution')
sns.histplot(np.log1p(pd.to_numeric(df[TARGET_COLUMN],errors='coerce').clip(lower=0).dropna()),kde=True,ax=ax[1])
ax[1].set_title('Log-Transformed Price')
plt.tight_layout(); plt.show()


In [ ]:
# Feature engineering: date parts and log versions of strongly right-skewed non-negative numeric predictors.
work=df.copy()
for col in list(work.columns):
    if 'date' in col.lower():
        parsed=pd.to_datetime(work[col],errors='coerce')
        if parsed.notna().mean()>0.5:
            work[col+'_year']=parsed.dt.year
            work[col+'_month']=parsed.dt.month
            work.drop(columns=[col],inplace=True)
num_pred=work.drop(columns=[TARGET_COLUMN],errors='ignore').select_dtypes(include=np.number)
skew=num_pred.skew(numeric_only=True).abs().sort_values(ascending=False)
for col in skew[skew>1].index:
    if work[col].min(skipna=True)>=0:
        work[col+'_log1p']=np.log1p(work[col])

X=work.drop(columns=[TARGET_COLUMN])
y=pd.to_numeric(work[TARGET_COLUMN],errors='coerce')
valid=y.notna(); X=X.loc[valid].copy(); y=y.loc[valid]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE)
numeric_features=X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features=X_train.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric features:',len(numeric_features))
print('Categorical features:',len(categorical_features))


In [ ]:
numeric_transformer=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_transformer=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocessor=ColumnTransformer([('num',numeric_transformer,numeric_features),('cat',categorical_transformer,categorical_features)],remainder='drop')
models={
 'Linear Regression':LinearRegression(),
 'Random Forest':RandomForestRegressor(n_estimators=300,random_state=RANDOM_STATE,n_jobs=-1),
 'Gradient Boosting':GradientBoostingRegressor(n_estimators=200,learning_rate=.05,max_depth=3,random_state=RANDOM_STATE)
}
results=[]; trained={}
for name,estimator in models.items():
    pipe=Pipeline([('preprocessor',preprocessor),('model',estimator)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    results.append({'Model':name,'RMSE':np.sqrt(mean_squared_error(y_test,pred)),'MAE':mean_absolute_error(y_test,pred),'R2':r2_score(y_test,pred)})
    trained[name]=pipe
results_df=pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
display(results_df)


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=results_df,x='Model',y='RMSE',ax=ax[0]); ax[0].tick_params(axis='x',rotation=20); ax[0].set_title('RMSE Comparison')
sns.barplot(data=results_df,x='Model',y='MAE',ax=ax[1]); ax[1].tick_params(axis='x',rotation=20); ax[1].set_title('MAE Comparison')
plt.tight_layout(); plt.show()

best_name=results_df.iloc[0]['Model']; best_pipeline=trained[best_name]; best_pred=best_pipeline.predict(X_test); residuals=y_test-best_pred
print('Best model:',best_name)
print('RMSE:',np.sqrt(mean_squared_error(y_test,best_pred)))
print('MAE:',mean_absolute_error(y_test,best_pred))
print('R2:',r2_score(y_test,best_pred))
fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(x=best_pred,y=residuals,ax=ax[0]); ax[0].axhline(0,linestyle='--'); ax[0].set_title('Residuals vs Predicted'); ax[0].set_xlabel('Predicted Price'); ax[0].set_ylabel('Residual')
sns.histplot(residuals,kde=True,ax=ax[1]); ax[1].set_title('Residual Distribution'); ax[1].set_xlabel('Residual')
plt.tight_layout(); plt.show()
MODEL_PATH.parent.mkdir(parents=True,exist_ok=True); joblib.dump(best_pipeline,MODEL_PATH); print('Saved:',MODEL_PATH)


In [ ]:
# Small inference example using one real test row, guaranteeing the same feature schema.
example_house=X_test.iloc[[0]].copy()
prediction=best_pipeline.predict(example_house)[0]
display(example_house)
print('Predicted house price:',prediction)
print('Actual house price:',y_test.iloc[0])
